# Agent Memory, Zero to Hero 🧠

### Building a memory‑augmented engineering copilot with **MemoRizz** + **Oracle AI Database**

LLMs are stateless — every API call starts from a blank slate. Real products need agents that
*remember*: the user, past conversations, documents, tools, runbooks, and each other. This notebook
builds that capability up **one memory type at a time**, so you can see exactly what each layer adds
and *why* you'd reach for it.

## What You Will Build

A single running example — **Memo**, an engineering copilot for an AI platform team — grown from a
goldfish into a colleague across twelve layers of memory, each adding exactly one capability:

1. **The problem** — a stateless LLM that forgets between calls (the motivation).
2. **Episodic memory** — conversation that persists across turns *and* process restarts.
3–5. **Semantic memory** — a stable persona, structured entity facts, and a knowledge base (RAG).
6–8. **Procedural memory** — tools, repeatable workflows, and a skillbox of how‑to guides.
9–10. **Short‑term / working memory** — a semantic cache and context‑window summaries.
11. **Shared memory** — multiple agents coordinating over a blackboard.
12. **Putting it together** — application modes and a fully memory‑augmented agent.

By the end you'll have a clear mental model of *which* memory type solves *which* problem — and the
exact MemoRizz API for each.

## What You Will Learn

- Persist a conversation across turns **and a process restart** with a `MemAgent` on Oracle
- Give an agent a durable **persona** and capture structured **entity** facts it can recall precisely
- Ground answers in your own documents with a **knowledge base** (RAG) over Oracle AI Vector Search
- Equip an agent with **tools**, recall **workflows** by intent, and load **skills** on demand
- Cut cost and latency with a **semantic cache**, and stay within budget using **summaries**
- Coordinate a **team of agents** over shared memory with `MultiAgentOrchestrator`
- Map the cognitive‑science memory taxonomy (episodic / semantic / procedural / working / social) onto
  MemoRizz `MemoryType`s and pick the right one for a given job

## The Memory Stack at a Glance

Cognitive science splits human memory into a few systems. Agent frameworks borrow the same
vocabulary, and MemoRizz implements each one as a `MemoryType` backed by a storage provider.

| Human memory | What the agent needs | `MemoryType` | MemoRizz API | Part |
|---|---|---|---|---|
| **Working memory** | Hold the active task in the context window | `SHORT_TERM_MEMORY` | `get_context_window_stats()` | 10 |
| *(engineering speed‑up)* | Don't recompute identical answers | `SEMANTIC_CACHE` | `enable_semantic_cache()` | 9 |
| **Episodic** | Remember past interactions | `CONVERSATION_MEMORY` | `agent.run()` (auto) | 2 |
| **Episodic** (compressed) | Keep long histories affordable | `SUMMARIES` | `generate_summaries()` | 10 |
| **Semantic** (identity) | A stable persona / voice | `PERSONAS` | `Persona`, `set_persona()` | 3 |
| **Semantic** (facts) | Structured profile facts | `ENTITY_MEMORY` | `EntityMemory` | 4 |
| **Semantic** (knowledge) | Long‑term documents / RAG | `KNOWLEDGE_BASE` | `KnowledgeBase.ingest_*()` | 5 |
| **Procedural** (skills) | *How* to act — tools | `TOOLBOX` | `Toolbox`, `with_tool()` | 6 |
| **Procedural** (processes) | Repeatable runbooks | `WORKFLOW_MEMORY` | `Workflow` | 7 |
| **Procedural** (guides) | Reusable skills to *read & follow* | *custom* `skillbox` | `build_skill_manifest()` / `load_skill()` | 8 |
| **Procedural** (audit) | What the agent actually did | `TOOL_LOG` | `memory_manager.list_tool_logs()` | 6 |
| **Social** | Multi‑agent coordination | `SHARED_MEMORY` | `SharedMemory`, `MultiAgentOrchestrator` | 11 |
| *(durability)* | Survive a process restart | `MEMAGENT` | `agent.save()` / `MemAgent.load()` | 2 |

![The agent memory stack — types of memory](images/agent_memory_types.png)


> 💡 **Key Insight — Add one memory type at a time:** Each part of this notebook differs from the one
> before it by a *single* new capability. Naming that difference is the whole point — it tells you the
> simplest memory a problem actually needs. Reaching for multi‑agent shared memory when a conversation
> store would do is the agent‑memory equivalent of over‑engineering.

> 📎 **Companion notebook:** This is the deep‑dive sibling of
> [`part3/memagent_local_oracle.ipynb`](../../part3/memagent_local_oracle.ipynb). That notebook gets
> you running on local Oracle fast; this one walks through **every** memory type — including a custom
> **Skillbox** (Part 8) — and *why* each one exists. Each part follows the same rhythm:
> **the problem → the concept → the code → key takeaways.**

---
## Setup · Provision the Memory Core

Memory needs somewhere durable to live. MemoRizz is **provider‑pluggable** (filesystem,
MongoDB, Oracle), and here we use **Oracle AI Database 23ai** as the *Memory Core*: one engine
that stores relational rows **and** does native **vector search** over embeddings — so every
memory type below (conversation, knowledge base, tools, cache, …) shares a single backing store.

The setup mirrors the companion notebook, so if you already have local Oracle running you can
skim to §0.4.

### 0.1 · Install

PyPI ships the same version as the source tree (`memorizz` ≥ `0.0.49`), so a normal install gets
every memory feature used here, plus `oracledb` (the Oracle driver), `openai`, and `python-dotenv`.

> 🛠️ **Developing against a local checkout of memorizz?** Install it editable instead, e.g.
> `%pip install -qU -e "/path/to/memorizz[oracle]"`.

In [1]:
%pip install -qU "memorizz[oracle]" oracledb openai python-dotenv
print("\n✅ Environment ready")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.14 requires openai==2.24.0, but you have openai 2.41.1 which is incompatible.
oracleagentmemory 26.4.0 requires oracledb<4,>=3.4.2, but you have oracledb 4.0.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.

✅ Environment ready


### 0.2 · Start & configure local Oracle

We use the MemoRizz CLI to pull and run Oracle AI Database Free in Docker. It is **idempotent** —
safe to re‑run. (Requires Docker Desktop running. On Apple Silicon the helper handles the
`linux/amd64` platform flag for you.)

If you have *never* run this before, uncomment and run the next cell; it takes a few minutes the
first time while it downloads the image (~1.8 GB) and waits for *"DATABASE IS READY TO USE!"*.

In [2]:
# First time only — provision the local Oracle container (uncomment to run):
# !memorizz install-oracle

### 0.3 · Connection & embedding environment variables

These defaults match the local container created above. The `MEMORIZZ_DEFAULT_EMBEDDING_*`
variables make every client (this notebook, the UI, the CLI) embed with the **same model and
dimension** — critical, because vectors created with different dimensions can't be compared.

In [3]:
import os

# Admin password for the local oracle-free container (used by setup_oracle_user to CREATE the
# user). Hard-set, not getenv: the kernel may hold a stale wrong value from an earlier run.
os.environ["ORACLE_ADMIN_PASSWORD"] = "OraclePwd_2025"
# Fresh, 256-dim schema. The earlier "memorizz_user" schema was built by previous runs at a
# different (2050-dim) embedding size; memorizz only CREATEs tables that don't already exist,
# so it can't be reused at 256. Point at a NEW user so every table is created clean at
# VECTOR(256). Hard-assigned (not getenv) so re-running this cell switches schemas even if the
# kernel still holds the old ORACLE_USER value.
os.environ["ORACLE_USER"]           = "memorizz_user_256"
os.environ["ORACLE_PASSWORD"]       = os.getenv("ORACLE_PASSWORD", "SecurePass123!")
os.environ["ORACLE_DSN"]            = os.getenv("ORACLE_DSN", "localhost:1521/FREEPDB1")

os.environ["MEMORIZZ_DEFAULT_EMBEDDING_PROVIDER"]   = "openai"
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_MODEL"]      = "text-embedding-3-small"
os.environ["MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS"] = "256"

print("Oracle DSN:   ", os.environ["ORACLE_DSN"])
print("Oracle schema:", os.environ["ORACLE_USER"])
print("Embeddings:   ", os.environ["MEMORIZZ_DEFAULT_EMBEDDING_MODEL"],
      "@", os.environ["MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS"], "dims")

Oracle DSN:    localhost:1521/FREEPDB1
Oracle schema: memorizz_user_256
Embeddings:    text-embedding-3-small @ 256 dims


### 0.4 · Your OpenAI key

Used for both the **LLM** (the agent's reasoning) and the **embeddings** (how every memory is
indexed for semantic search).

In [4]:
import os, getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
print("OPENAI_API_KEY set:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


### 0.5 · Create the schema

`setup_oracle_user()` creates the `memorizz_user`, grants the AI Vector Search privileges, and
runs the schema migrations that create one table per memory type (`CONVERSATION_MEMORY`,
`PERSONAS`, `TOOLBOX`, `KNOWLEDGE_BASE`, `SHARED_MEMORY`, …). Safe to re‑run.

In [5]:
from memorizz.memory_provider.oracle.setup import setup_oracle_user

setup_oracle_user()
print("\n✅ Schema ready")

Oracle Database Complete Setup for Memorizz

✓ Found schema file: schema_relational.sql

Detecting setup mode...
----------------------------------------------------------------------
  Admin user 'system' cannot create users
  Trying SYS as SYSDBA (has full privileges)...
  ✓ Connected as SYS as SYSDBA (has CREATE USER privilege)
✓ Admin mode detected: Full setup with user creation
  Connected as: sys
  Can create users: Yes

STEP 1: Creating User and Granting Privileges
----------------------------------------------------------------------

Dropping existing memorizz_user_256 user (if exists)...
  Checking for active sessions...
  No active sessions found
  ✓ Dropped existing memorizz_user_256 user

Creating memorizz_user_256 user...
  ✓ User memorizz_user_256 created

Granting basic privileges (least-privilege)...
  ✓ CREATE SESSION (required for database connections)
  ✓ CREATE TABLE (required for memory storage tables and indexes)
  ✓ CREATE VIEW (required for Memorizz views)
  ✓ 

### 0.6 · Build the Memory Core (the Oracle provider)

The `OracleProvider` is the single object every memory type plugs into. Constructing it
auto‑creates any missing tables and (because `lazy_vector_indexes=False`) builds the vector
indexes up front.

In [6]:
import os
from memorizz.memory_provider.oracle import OracleProvider, OracleConfig

oracle_config = OracleConfig(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASSWORD"],
    dsn=os.environ["ORACLE_DSN"],
    schema=os.environ["ORACLE_USER"],
    lazy_vector_indexes=False,
    embedding_provider="openai",
    embedding_config={
        "model": os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_MODEL", "text-embedding-3-small"),
        "dimensions": int(os.getenv("MEMORIZZ_DEFAULT_EMBEDDING_DIMENSIONS", "256")),
        "api_key": os.environ["OPENAI_API_KEY"],
    },
)

memory_core = OracleProvider(oracle_config)
print("🧠 Memory Core online — Oracle provider ready")

# One LLM config we'll reuse for every agent in this notebook.
LLM = {"provider": "openai", "model": "gpt-5.5", "api_key": os.environ["OPENAI_API_KEY"]}

Failed to create vector index idx_tool_log_vec: ORA-00904: "EMBEDDING": invalid identifier
Help: https://docs.oracle.com/error-help/db/ora-00904/


🧠 Memory Core online — Oracle provider ready


*(Optional)* Turn on INFO logging to watch memory being retrieved and written on each turn —
great for understanding what the agent loop actually does.

In [7]:
import logging, os
os.environ["MEMORIZZ_LOG_LEVEL"] = "WARNING"   # set to "INFO" to see memory reads/writes
logging.basicConfig(level=logging.WARNING, format="%(levelname)s · %(name)s · %(message)s", force=True)

---
# Part 1 · The problem — an amnesiac agent 🐟

> 💡 **Key Term — Statelessness:** An LLM API call is independent — the model remembers nothing between calls. Any "memory" is something you (or a memory system) place back into the prompt.

Before adding memory, let's feel its absence. A raw LLM call has **no state**: anything you say
in one request is gone by the next. Watch it forget who you are between two back‑to‑back calls.

In [8]:
from openai import OpenAI

client = OpenAI()

def stateless_ask(prompt: str) -> str:
    resp = client.chat.completions.create(
        model="gpt-5.5",
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content

print("Turn 1 →", stateless_ask("Hi, I'm Ada. I'm migrating our RAG stack to Oracle 23ai vector search."))
print("\nTurn 2 →", stateless_ask("What database am I migrating to?"))

Turn 1 → Hi Ada! Nice—Oracle 23ai vector search can be a solid fit for RAG, especially if your data already lives in Oracle.

I can help with things like:

- Designing the schema for documents/chunks/embeddings
- Choosing `VECTOR` dimensions and distance metrics
- Loading embeddings into Oracle 23ai
- Creating vector indexes
- Writing similarity search SQL
- Hybrid search patterns with metadata filters / text search
- RAG retrieval query design
- Migration planning from Pinecone, FAISS, pgvector, Elasticsearch, etc.
- Python integration examples

A typical Oracle 23ai RAG table might look roughly like:

```sql
CREATE TABLE rag_chunks (
    id           NUMBER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    doc_id       VARCHAR2(100),
    chunk_text   CLOB,
    metadata     JSON,
    embedding    VECTOR(1536, FLOAT32)
);
```

And a basic similarity query:

```sql
SELECT
    id,
    doc_id,
    chunk_text,
    VECTOR_DISTANCE(embedding, :query_embedding, COSINE) AS distance
FROM rag_ch

**What just happened:** Turn 2 has no idea — the model never saw turn 1. You *could* manually
re‑send the whole transcript every time, but that breaks down fast: it doesn't survive a restart,
it blows the context window on long histories, it can't recall a document from last month, and it
can't share state with another agent. Those are exactly the gaps the memory types below close.

> ### 📌 Key Takeaways — Part 1
> - A raw LLM call is **stateless** — it forgets everything between requests.
> - Re‑sending the transcript yourself works briefly but breaks on restarts, long histories, document recall, and multi‑agent sharing.
> - Those gaps are exactly what the memory layers ahead close, one at a time.

---
# Part 2 · Episodic memory I — persistent conversation 💬

> 💡 **Key Term — Episodic memory (`CONVERSATION_MEMORY`):** The time‑stamped record of interactions — the agent's autobiography. `MemAgent.run()` retrieves the relevant history, reasons, and writes the new turns back automatically.

> ➕ **Adding:** `CONVERSATION_MEMORY` + `MEMAGENT` &nbsp;·&nbsp; *the agent remembers what was said —
> even after the process dies.*

**Episodic memory** is the autobiographical record of interactions: who said what, when. A
`MemAgent` backed by our Memory Core does this automatically. On every `run()` it:

1. **retrieves** relevant prior turns (recent history + semantically similar older turns),
2. **reasons** with the LLM, then
3. **persists** the new user/assistant turns back to `CONVERSATION_MEMORY`.

Calling `agent.save()` also writes the agent's *configuration* to the `MEMAGENT` store, so the
whole agent can be reconstructed later by its `agent_id`.

In [9]:
from memorizz.memagent.builders import MemAgentBuilder

copilot = (
    MemAgentBuilder()
    .with_name("Memo")
    .with_instruction(
        "You are Memo, an engineering copilot for an AI platform team. "
        "Be concise, concrete, and technical."
    )
    .with_memory_provider(memory_core)
    .with_llm_config(LLM)
    .build()
)

In [10]:
copilot.save()                      # persist the agent's config to the MEMAGENT store
AGENT_ID = copilot.agent_id         # remember this — it's how we reload the agent later
print("Created agent:", copilot.name, "| agent_id =", AGENT_ID)

Created agent: Memo | agent_id = 2e82a8a7-aabf-4418-94fa-e336af26485a


In [11]:
print(copilot.run("Hi Memo, I'm Ada. I'm migrating our RAG stack to Oracle 23ai vector search."))

Hi Ada — got it. I’ll keep Oracle 23ai vector search in mind for this RAG migration.

I can help with schema design, embedding storage, HNSW/IVF index choices, hybrid search, chunking strategy, query plans, migration scripts, benchmarking, or app-layer retrieval changes.


In [12]:
print(copilot.run("Remind me — what am I migrating to, and who am I?"))

# Re-persist AFTER the conversation: save() stores the agent's memory_ids, and the active
# conversation thread's memory_id only exists after these runs. Without this, a fresh
# MemAgent.load() restores the agent config but not this conversation.
copilot.save()

You’re Ada, and you’re migrating your RAG stack to Oracle 23ai vector search.


Now the agent remembers within the session. But the real test of *persistent* memory is
surviving a **process restart**. Let's simulate a brand‑new Python session by reconstructing the
agent from scratch using only its `agent_id` — no in‑memory state carried over.

In [13]:
from memorizz.memagent import MemAgent

# Imagine the kernel was restarted. We only kept the agent_id string.
reloaded = MemAgent.load(AGENT_ID, memory_provider=memory_core)

print(reloaded.run("Quick recap with no extra context: what project of mine were we discussing?"))

Migrating your RAG stack to Oracle 23ai vector search.


**What just happened:** A *freshly loaded* agent answered correctly because the conversation
lives in Oracle, not in Python memory. This is the difference between a chatbot and a colleague.

**Threads & scopes.** Each agent owns one or more `memory_id`s (conversation scopes); within a
scope, `thread_id`s separate distinct conversations. Use `start_new_thread()` for a clean topic
and `switch_thread()` to jump back. Pass `user_id=...` to `run()` to isolate memory per end‑user
when one agent serves many people.

> ### 📌 Key Takeaways — Part 2
> - A `MemAgent` on a provider remembers across turns **and across a process restart** (`MemAgent.load(agent_id)`).
> - `run()` auto‑retrieves relevant history and auto‑persists each turn — you never manage the transcript.
> - Threads (`memory_id` / `thread_id`) and `user_id` scope memory per conversation and per end‑user.

---
# Part 3 · Semantic memory I — persona 🎭

> 💡 **Key Term — Persona (`PERSONAS`):** A durable identity — name, role, goals, background — that's *versioned*: every change is appended to an auditable `evolution_history`.

> ➕ **Adding:** `PERSONAS` &nbsp;·&nbsp; *a stable identity, role, and voice that persists and can evolve.*

**Semantic memory** stores durable facts and identity rather than time‑stamped events. The first
slice is the agent's **persona**: its name, role, goals, and background. A persona is *versioned* —
every change is appended to an `evolution_history` with a `change_trigger`, so identity drift is
auditable. Personas live in the `PERSONAS` store.

In [14]:
from memorizz import Persona, RoleType

# RoleType options: GENERAL, ASSISTANT, CUSTOMER_SUPPORT, TECHNICAL_EXPERT, RESEARCHER
persona = Persona(
    name="Memo",
    role=RoleType.TECHNICAL_EXPERT,
    goals="Help engineers ship reliable LLM applications; favor correctness first, then clarity.",
    background=(
        "A staff-level AI platform engineer with deep experience in retrieval, vector "
        "databases, evaluation, and running agents in production."
    ),
)


In [15]:
# Attach to our existing copilot (persists to PERSONAS and refreshes the agent's snapshot).
copilot.set_persona(persona)
print(copilot.run("Who are you, and how should I expect you to help me?"))

I’m Memo — a technical expert / engineering copilot for an AI platform team.

You should expect me to help you ship reliable LLM applications with concrete, correctness-first technical guidance, especially around retrieval, vector databases, evaluation, and production agents. For your current project, I can help with Oracle 23ai vector search migration details like schema/index design, hybrid retrieval, benchmarking, query tuning, and RAG evaluation.


**What just happened:** The agent now answers *in character* and will keep that identity on
every future turn and reload. Because a `MemAgent` with a persona automatically gains
`read_persona` and `update_persona` tools, it can even **evolve** its own identity when justified —
each edit recorded with the reason that triggered it. Use persona for *durable identity*; use
entity memory (next) for *one‑off facts*.

> ### 📌 Key Takeaways — Part 3
> - A persona gives the agent a **stable, consistent voice and role** that survives reloads.
> - Personas are **versioned** — every change is recorded with the trigger that caused it.
> - Use persona for *durable identity*; use entity memory (next) for *one‑off facts*.

---
# Part 4 · Semantic memory II — entity memory 🗂️

> 💡 **Key Term — Entity memory (`ENTITY_MEMORY`):** Structured profiles — entities with typed **attributes** and **relations** — a lightweight, agent‑maintained knowledge graph for precise factual recall.

> ➕ **Adding:** `ENTITY_MEMORY` &nbsp;·&nbsp; *structured, queryable facts about people, services, and systems.*

Conversation memory is great at "what was said" but bad at "what is *true*." **Entity memory**
stores structured profiles — entities with typed **attributes** and **relations** — so the agent
can answer precise questions ("who owns the vector‑search service?") without re‑reading the whole
transcript. Think of it as a lightweight, agent‑maintained knowledge graph.

You can drive it two ways. First, **programmatically**, for facts you control deterministically:

In [16]:
from memorizz.long_term.semantic.entity_memory.entity_memory import EntityMemory

em = EntityMemory(memory_core)

# Store entities in the SAME memory scope the agent queries under. memorizz scopes semantic
# entity lookups by memory_id, so entities written with a different (or no) memory_id are
# invisible to an agent's entity_memory_lookup. Reusing copilot's memory_id keeps Ada & co.
# retrievable by copilot now — and by Memo-Pro at the end (it shares this same scope).
_MID = copilot.memory_ids[0] if copilot.memory_ids else None

ada_id = em.upsert_entity(
    name="Ada",
    entity_type="person",
    attributes=[
        {"name": "role", "value": "Senior ML Engineer"},
        {"name": "current_project", "value": "RAG migration to Oracle 23ai"},
        {"name": "preferred_language", "value": "Python"},
    ],
    memory_id=_MID,
)

svc_id = em.upsert_entity(
    name="vector-search-svc",
    entity_type="service",
    attributes=[
        {"name": "datastore", "value": "Oracle 23ai"},
        {"name": "sla", "value": "99.9%"},
        {"name": "embedding_model", "value": "text-embedding-3-small (256d)"},
    ],
    memory_id=_MID,
)

# Relations turn isolated facts into a graph.
em.link_entities(source_entity_id=ada_id, target_entity_id=svc_id, relation_type="owns")

print("Ada's profile:")
print(em.get_entity_profile(ada_id))

Ada's profile:
{'entity_id': '45f5098c-1c88-4249-b1c9-efc4fd785f06', 'name': 'Ada', 'entity_type': 'person', 'attributes': {'name': 'Ada', 'current_project': 'RAG migration to Oracle 23ai', 'role': 'Senior ML Engineer', 'preferred_language': 'Python'}, 'updated_at': datetime.datetime(2026, 6, 16, 12, 51, 57, 784995), 'relations': [{'entity_id': '9412917f-a861-4213-82f4-91f66b3aa23d', 'relation_type': 'owns', 'confidence': 0.7}]}


In [17]:
# Semantic search over entities — natural-language question, structured answer.
for hit in em.search_entities("which service runs on Oracle and who owns it?", limit=3):
    print("•", hit.get("name"), "—", hit.get("entity_type"))

• vector-search-svc — service
• Ada — person


Second, you can let the **agent capture facts itself**. Enabling entity memory gives the LLM
`entity_memory_upsert` / `entity_memory_lookup` tools, so it records salient facts mid‑conversation
and retrieves them later.

In [18]:
copilot.with_entity_memory(True)   # agent can now read/write structured facts on its own

# Record the fact as an ENTITY + ATTRIBUTE (not a relation). memorizz's relation tool-arg shape
# is under-documented, so models often mis-call it (EntityRelation validation error); storing
# the owner as an attribute is reliable and lands in copilot's entity-memory scope.
copilot.run("For the record: record an entity named PagerPilot (type: service) with an attribute 'owner' set to 'Ada'.")
print(copilot.run("Who owns PagerPilot? Answer from your entity memory."))

Ada owns PagerPilot.


**What just happened:** Facts are now first‑class, structured, and queryable independent of
when they were mentioned. Reach for entity memory when you need **precise recall of attributes**
(ownership, config, preferences); reach for the knowledge base (next) when you need **passages
from documents**.

> ### 📌 Key Takeaways — Part 4
> - Entity memory stores **structured, queryable facts** (ownership, config, preferences) independent of when they were mentioned.
> - The agent can capture facts itself (`with_entity_memory`) or you can upsert them deterministically (`EntityMemory`).
> - Reach for entities for *precise attribute recall*; reach for the knowledge base for *document passages*.

---
# Part 5 · Semantic memory III — knowledge base (RAG) 📚

> 💡 **Key Term — Knowledge base (`KNOWLEDGE_BASE`):** Long‑term document memory: chunk → embed → vector‑search → ground the answer. This is RAG, living inside the agent's memory system.

> ➕ **Adding:** `KNOWLEDGE_BASE` &nbsp;·&nbsp; *long‑term document memory via chunking + vector retrieval —
> i.e. RAG, living inside the agent's memory system.*

The third slice of semantic memory is a **knowledge base**: ingest documents, chunk them, embed
each chunk, and retrieve the most relevant passages at query time. Attaching a KB to an agent
gives it a `knowledge_base_lookup` tool scoped to exactly that knowledge.

We'll ingest a short (synthetic) platform runbook. `ingest_knowledge` chunks and embeds it; here
we chunk by paragraph.

In [19]:
from memorizz import KnowledgeBase

kb = KnowledgeBase(memory_core)

runbook = """Acme RAG Platform — Engineering Runbook

Embedding policy. All production text is embedded with text-embedding-3-small at 256 dimensions.
Never mix dimensions within an index: vectors of different sizes are not comparable and retrieval
will silently degrade. Re-embed the entire corpus if you change the model or dimension.

Rebuilding the vector index. To rebuild: (1) put the service in maintenance mode, (2) run
`acme reindex --service vector-search-svc --batch 500`, (3) wait for the row counts in the new
and old indexes to match, (4) flip the alias, (5) leave maintenance mode. A full rebuild on the
current corpus takes about 40 minutes.

Chunking. Default chunking is paragraph-based with a 1000-character cap. Long tables are extracted
separately and stored with a `table` namespace so they are retrieved as whole units.

On-call. Page severity SEV1 means user-facing retrieval is down. The on-call engineer must
acknowledge within 5 minutes via PagerPilot and post status to #incidents every 30 minutes.
"""

kb_id = kb.ingest_knowledge(corpus=runbook, namespace="platform-runbook", chunking_strategy="paragraph")

In [20]:
kb.attach_to_agent(copilot, kb_id)
print("Ingested knowledge base:", kb_id)

Ingested knowledge base: b825ec35-b4e0-4acc-a58e-3594b33120e4


In [21]:
# Direct retrieval — see the raw passages the agent would ground on.
hits = kb.retrieve_knowledge_by_query("how do we rebuild the vector index?", namespace="platform-runbook", limit=2)
for i, h in enumerate(hits, 1):
    print(f"[{i}] {h.get('content', '')[:220].strip()} ...\n")

[1] Rebuilding the vector index. To rebuild: (1) put the service in maintenance mode, (2) run
`acme reindex --service vector-search-svc --batch 500`, (3) wait for the row counts in the new
and old indexes to match, (4) flip ...

[2] Embedding policy. All production text is embedded with text-embedding-3-small at 256 dimensions.
Never mix dimensions within an index: vectors of different sizes are not comparable and retrieval
will silently degrade. Re ...



In [22]:
# The agent now answers grounded in the runbook, via its knowledge_base_lookup tool.
print(copilot.run("According to our runbook, what's the exact procedure to rebuild the vector index?"))

According to the runbook, rebuild the vector index as follows:

1. Put the service in maintenance mode.
2. Run:
   ```bash
   acme reindex --service vector-search-svc --batch 500
   ```
3. Wait for row counts in the new and old indexes to match.
4. Flip the alias.
5. Leave maintenance mode.

Expected duration for the current corpus: about 40 minutes.


**What just happened:** The agent answered from *your* documents, not its pretraining. For
real corpora you rarely paste strings — use `kb.ingest_file("docs/handbook.pdf", namespace=...)`
or `kb.ingest_directory("docs/", recursive=True, extensions=[".pdf", ".md"])`. Text formats work
out of the box; add `pip install "memorizz[ingest-pdf]"` for PDFs. Chunking strategies:
`fixed` (default), `sentence`, `paragraph`, `semantic`, or a custom callable.

> ### 📌 Key Takeaways — Part 5
> - A `KnowledgeBase` makes the agent answer from **your documents**, with citations — not its pretraining.
> - Ingest strings, files, or whole folders; pick a chunking strategy (`fixed` / `sentence` / `paragraph` / `semantic`).
> - Retrieval quality is the single biggest lever on answer quality.

---
# Part 6 · Procedural memory I — the toolbox 🛠️

> 💡 **Key Term — Procedural memory · Toolbox (`TOOLBOX`):** *How* to act — Python callables the LLM can discover and invoke. A function's type hints and docstring **are** its tool spec.

> ➕ **Adding:** `TOOLBOX` + `TOOL_LOG` &nbsp;·&nbsp; *the agent learns **how to act** — tools it can
> discover, call, and have audited.*

**Procedural memory** is "how to do things." Its first form is **tools**: Python callables the LLM
can invoke. MemoRizz introspects each function's signature and docstring to build the schema the
model sees — so good type hints and docstrings *are* the tool spec.

The quickest path is `with_tool()` for tools that live with one agent:

In [23]:
def estimate_monthly_cost(num_requests: int, tokens_per_request: int,
                          usd_per_million_tokens: float = 0.15) -> float:
    """Estimate monthly LLM token cost (USD) from request volume and average tokens per request."""
    return round(num_requests * tokens_per_request / 1_000_000 * usd_per_million_tokens, 2)

cost_agent = (
    MemAgentBuilder()
    .with_instruction("You estimate LLM costs. Always use the provided tool for any calculation.")
    .with_tool(estimate_monthly_cost)
    .with_memory_provider(memory_core)
    .with_llm_config(LLM)
    .build()
)
print(cost_agent.run(
    "We expect ~2,000,000 requests/month at about 800 input tokens each on gpt-4o-mini. "
    "Roughly what's the monthly input-token cost?"
))

Roughly **$240/month** for input tokens.

Calculation:  
- 2,000,000 requests × 800 input tokens = **1.6B input tokens/month**
- gpt-4o-mini input pricing ≈ **$0.15 per 1M tokens**
- 1,600 × $0.15 = **$240**


For tools you want to **persist, share, and retrieve semantically**, register them in a
`Toolbox`. Each tool is stored (with embeddings) in the `TOOLBOX` memory, so at scale an agent can
*search* for the right tool by intent instead of being handed all of them.

In [24]:
from memorizz import Toolbox
from memorizz.llms.openai import OpenAI as _MemOpenAILLM
from pydantic import BaseModel
import inspect

# memorizz's default tool-metadata path asks an LLM (responses.parse) to generate the tool's
# name/description, and it frequently returns name=None -> tools get stored as "unknown_tool".
# The function name is authoritative, so use a provider that fills metadata deterministically.
class _FlatToolMeta(BaseModel):
    name: str
    description: str = ""
    signature: str = ""
    docstring: str = ""
    type: str = "function"

class NamedToolOpenAI(_MemOpenAILLM):
    def get_tool_metadata(self, func):
        doc = (func.__doc__ or "").strip()
        return _FlatToolMeta(
            name=func.__name__,
            description=doc or func.__name__,
            signature=str(inspect.signature(func)),
            docstring=doc,
            type="function",
        )

toolbox = Toolbox(memory_core, llm_provider=NamedToolOpenAI(), agent_id=copilot.agent_id)

def check_service_health(service_name: str) -> dict:
    """Return the current health status for a named internal service."""
    return {"service": service_name, "status": "healthy", "p99_latency_ms": 240}

def list_recent_deploys(service_name: str, limit: int = 5) -> list:
    """List the most recent deployments for a service, newest first."""
    return [{"service": service_name, "version": f"v1.4.{i}", "status": "success"} for i in range(limit)]

# register_tool() stores the function (with embeddings) and returns the stored tool's ID.
# It also works as `@toolbox.register_tool`, but note the decorator rebinds the name to the
# returned ID string — so use a plain def + register_tool() if you also call the tool locally.
health_id  = toolbox.register_tool(check_service_health)
deploys_id = toolbox.register_tool(list_recent_deploys)
print("Registered tool IDs:", health_id, deploys_id)

print("Tools in the toolbox:", [t.get("name") for t in toolbox.list_tools()])

Registered tool IDs: 3e37f284-1f86-434f-a354-830c6c349153 8c44f74e-2f8b-408d-b506-38ff2f908173
Tools in the toolbox: ['automation_resume_job', 'automation_delete_job', 'automation_run_now', 'context_window_stats_tool', 'list_recent_tool_logs', 'knowledge_base_lookup', 'list_summary_registry_tool', 'expand_summary', 'summarize_conversation', 'retrieve_tool_log_entry', 'automation_create_job', 'automation_list_jobs', 'automation_pause_job', 'entity_memory_lookup', 'entity_memory_upsert', 'update_persona', 'read_persona', 'check_service_health', 'list_recent_deploys']


In [25]:
# Semantic tool retrieval: find the right tool from a natural-language need.
matches = toolbox.get_most_similar_tools("is the API up right now?", limit=1)
print("Best match for 'is the API up?':", [t.get("name") for t in matches])

Best match for 'is the API up?': ['check_service_health']


You can register the same way with a **decorator**, and switch on **`augment=True`** to make a tool more *discoverable*. Augmentation asks the toolbox's LLM to enrich the tool's docstring and generate a handful of **synthetic queries** — example phrasings a user might use — then embeds all of it alongside the name and signature. The payoff: the agent finds the tool even when the request shares none of its wording. It costs a couple of extra LLM calls at registration, so reserve it for tools whose intent gets phrased many different ways.

In [26]:
# Decorator form, with augmentation turned on.
# Gotcha (same as the comment above): the decorator returns the stored tool's *ID string*, so
# the name `get_error_budget` is rebound to that ID — capture it, but don't call it locally.
@toolbox.register_tool(augment=True)
def get_error_budget(service_name: str) -> dict:
    """Return the remaining monthly error budget for a service, as a percentage."""
    return {"service": service_name, "error_budget_remaining_pct": 87.5, "window": "30d"}

print("Augmented tool ID:", get_error_budget)   # this is the ID string now, not a function

# Thanks to the synthetic queries, it's retrievable from phrasings that never say "error budget":
hit = toolbox.get_most_similar_tools("how much SLO headroom do we have left this month?", limit=1)
print("Best match for an SLO-phrased need:", [t.get("name") for t in hit])

Augmented tool ID: 42537e76-787f-4df4-8c12-db4fa59c5294
Best match for an SLO-phrased need: ['get_error_budget']


Every tool call an agent makes is also written to the **`TOOL_LOG`** — a procedural audit
trail you can inspect or replay. (Wrapped defensively: it's empty until a tool actually runs.)

In [27]:
try:
    mem_id = copilot.get_current_memory_id()
    logs = copilot.memory_manager.list_tool_logs(memory_id=mem_id, limit=5)
    if logs:
        for entry in logs:
            print("•", entry.get("tool_name"), "·", entry.get("timestamp"))
    else:
        print("No tool-call log entries yet for this agent's current scope.")
except Exception as exc:
    print("Tool-log view is optional here:", exc)

• knowledge_base_lookup · 2026-06-16T12:52:11
• entity_memory_upsert · 2026-06-16T12:52:00
• entity_memory_upsert · 2026-06-16T12:51:41


**What just happened:** The agent gained *skills*. `with_tool()` is perfect for a handful of
tools; a `Toolbox` scales to many by making tools **searchable** and **durable**, and the
`TOOL_LOG` keeps every invocation auditable.

> ### 📌 Key Takeaways — Part 6
> - `with_tool()` attaches a tool to one agent; a `Toolbox` makes tools **persistent and semantically retrievable** at scale.
> - MemoRizz introspects each function's signature + docstring to build the schema the model sees.
> - Every tool call is recorded in the **`TOOL_LOG`** for audit and replay.

---
# Part 7 · Procedural memory II — workflows 📋

> 💡 **Key Term — Workflow memory (`WORKFLOW_MEMORY`):** Repeatable, multi‑step procedures (runbooks) the agent can store and recall by intent — institutional "how we do things here."

> ➕ **Adding:** `WORKFLOW_MEMORY` &nbsp;·&nbsp; *repeatable, multi‑step procedures the agent can store
> and recall — institutional "how we do things here."*

A single tool is one action; a **workflow** is a remembered *sequence* of steps — a runbook. Stored
in `WORKFLOW_MEMORY`, workflows are retrievable by natural‑language intent, so an agent facing an
outage can recall *the* incident procedure instead of improvising.

In [28]:
from memorizz.long_term.procedural.workflow.workflow import Workflow

incident = Workflow(
    name="incident_response",
    description="Standard procedure for responding to a production incident in the RAG platform.",
    agent_id=copilot.agent_id,
)
incident.add_step("acknowledge", {"action": "Acknowledge the SEV page in PagerPilot within 5 min", "owner": "on-call"})
incident.add_step("triage",      {"action": "Run check_service_health on affected services", "tool": "check_service_health"})
incident.add_step("mitigate",    {"action": "If error rate > 5%, roll back to last good deploy", "tool": "list_recent_deploys"})
incident.add_step("communicate", {"action": "Post status to #incidents every 30 minutes"})
incident.add_step("postmortem",  {"action": "File a blameless postmortem within 48 hours"})

wf_id = incident.store_workflow(memory_core)
print("Stored workflow:", wf_id)

Stored workflow: e822bcf9-9746-4af9-94da-d7a6d107007b


In [29]:
# Recall the right runbook by describing the situation.
matches = Workflow.retrieve_workflows_by_query("what's our process when the service goes down?", memory_core, limit=1)
for w in matches:
    steps = (w.to_dict().get("steps") or {})
    print(f"Recalled '{w.name}': {list(steps.keys())}")

Recalled 'incident_response': ['acknowledge', 'triage', 'mitigate', 'communicate', 'postmortem']


### Capturing a workflow the agent *actually ran*

The runbook above was authored by hand. Workflow memory also records execution **automatically**: give an agent `application_mode="workflow"` and some tools, and every tool call it makes during a `run()` is captured as an ordered step — the arguments, the result, and the overall outcome — in `WORKFLOW_MEMORY`. You can then retrieve that trace by intent and replay exactly what the agent did. Here we hand **Memo-Runner** the same two tools we registered in the toolbox and let it work a triage request.

In [30]:
from memorizz.memagent.builders import MemAgentBuilder

# application_mode="workflow" turns on WORKFLOW_MEMORY (+ TOOLBOX), so each tool call this agent
# makes during run() is recorded as a step. We attach the same two functions we registered in
# the toolbox above so it has real tools to execute.
runner = (
    MemAgentBuilder()
    .with_name("Memo-Runner")
    .with_instruction(
        "You are an on-call engineer. Use your tools to gather facts before answering: when "
        "asked about a service, check its health AND its recent deploys, then give a short "
        "rollback recommendation grounded in what the tools returned."
    )
    .with_application_mode("workflow")
    .with_tools([check_service_health, list_recent_deploys])
    .with_memory_provider(memory_core)
    .with_llm_config(LLM)
    .build()
)
runner.save()

try:
    answer = runner.run(
        "vector-search-svc looks slow. Check its health and its recent deploys, "
        "then tell me whether we should roll back."
    )
    print("Agent answer:\n", answer)
except Exception as exc:
    print("Run skipped/failed (it makes live tool + LLM calls):", exc)

Agent answer:
 `vector-search-svc` is currently **healthy**, but p99 latency is **240 ms**, which supports the “looks slow” concern.

Recent deploys all show **successful**:
- `v1.4.4`
- `v1.4.3`
- `v1.4.2`
- `v1.4.1`
- `v1.4.0`

**Rollback recommendation:** I would **not roll back immediately** based only on this data. The service is healthy and there’s no failed deploy signal. I’d first compare latency before/after the latest deploy and check error rate/saturation. If the latency regression started after `v1.4.4`, then roll back to `v1.4.3`; otherwise treat it as a performance incident rather than a deploy rollback.


In [31]:
from memorizz.long_term.procedural.workflow.workflow import Workflow

# The run() above auto-recorded a "Tool Execution for Query" workflow into WORKFLOW_MEMORY.
# Retrieve it by intent and replay the exact steps the agent took.
hits = Workflow.retrieve_workflows_by_query(
    "checked a service's health and recent deploys to decide on a rollback",
    memory_core, limit=3,
)
# Prefer the auto-captured execution trace over the hand-authored runbook of the same topic.
captured = next((w for w in hits if w.name.startswith("Tool Execution")), hits[0] if hits else None)

if captured is None:
    print("No workflow captured yet — re-run the cell above so the agent makes tool calls.")
else:
    d = captured.to_dict()
    steps = d.get("steps") or {}
    print(f"Recalled '{d.get('name')}'  ·  outcome={d.get('outcome')}  ·  {len(steps)} step(s):")
    for label, info in steps.items():
        print(f"  • {label}")
        print(f"      args   = {info.get('arguments')}")
        print(f"      result = {info.get('result')}")

Recalled 'Tool Execution for Query'  ·  outcome=success  ·  2 step(s):
  • Step 1: check_service_health
      args   = {'service_name': 'vector-search-svc'}
      result = {'service': 'vector-search-svc', 'status': 'healthy', 'p99_latency_ms': 240}
  • Step 2: list_recent_deploys
      args   = {'service_name': 'vector-search-svc', 'limit': 5}
      result = [{'service': 'vector-search-svc', 'version': 'v1.4.0', 'status': 'success'}, {'service': 'vector-search-svc', 'version': 'v1.4.1', 'status': 'success'}, {'service': 'vector-search-svc', 'version': 'v1.4.2', 'status': 'success'}, {'service': 'vector-search-svc', 'version': 'v1.4.3', 'status': 'success'}, {'service': 'vector-search-svc', 'version': 'v1.4.4', 'status': 'success'}]


**What just happened:** The agent now carries **institutional procedure**, not just isolated
actions. Building an agent with `application_mode="workflow"` wires `WORKFLOW_MEMORY` + `TOOLBOX`
together so the agent plans against stored procedures and executes them with its tools.

> ### 📌 Key Takeaways — Part 7
> - A `Workflow` is a remembered **sequence of steps** — a runbook — recalled by natural‑language intent.
> - It carries *institutional procedure*, not just isolated actions.
> - `application_mode="workflow"` wires `WORKFLOW_MEMORY` + `TOOLBOX` together for plan‑and‑execute agents.

---
# Part 8 · Procedural memory III — the Skillbox 🧩

> 💡 **Key Term — Skillbox (progressive disclosure):** A memory of reusable **skills** (how‑to guides). An always‑on *manifest* lists the top‑k relevant skills; the agent calls `load_skill(name)` to pull a full guide only when it's needed.

> ➕ **Adding:** a custom **`skillbox`** &nbsp;·&nbsp; *a memory of reusable **skills** — how‑to guides the
> agent discovers and follows on demand.*

A **tool** is one executable action; a **workflow** is a fixed step sequence; the **knowledge base**
returns reference passages. A **skill** is different: a self‑contained *playbook* written for the
agent to **read and follow** ("how we discover an unfamiliar schema", "how we rebuild a vector
index safely"). The design pattern is **progressive disclosure**:

1. An always‑on **manifest** — the top‑k most relevant skill *names + one‑line descriptions* — is
   prepended to every turn, so the agent always knows *what skills exist* (cheap: a few lines).
2. When a listed skill is relevant, the agent calls **`load_skill(name)`** to pull the *full body*
   only then (expensive content, loaded just‑in‑time).

This keeps the context window small while giving the agent access to *hundreds* of skills. Crucially,
the skillbox is **not** a built‑in MemoRizz `MemoryType` — but we don't drop to raw SQL for it either.
We build it on memorizz's **`KnowledgeBase`** primitive, which shows memory types are **composable**:
a skillbox is just a knowledge base of how‑to guides, retrieved by intent.

> This mirrors the **Skillbox** in Oracle's
> [enterprise‑data‑agent‑harness workshop](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/workshops/enterprise-data-agent-harness-workshop/workshop/notebook_complete.ipynb),
> whose skillbox is seeded from [`oracle/skills`](https://github.com/oracle/skills). Below we build it
> on memorizz's **`KnowledgeBase`** primitive (embeddings + Oracle vector search via the same
> provider); §8.5 then shows the lower‑level, framework‑free **in‑database ONNX** version verbatim.

### 8.1 · Back the skillbox with `KnowledgeBase`

A skillbox is just "named guides, retrieved by similarity" — which is exactly what memorizz's
`KnowledgeBase` does (chunk → embed → vector‑search), through the **same Oracle provider and
embedding config** as the rest of the notebook. We give skills their own `namespace` so they never
mix with the §5 runbook, and store each skill as a single document (`chunking_strategy="none"`).

In [32]:
import json
from memorizz import KnowledgeBase

skill_kb = KnowledgeBase(memory_core)   # same provider -> same embeddings + Oracle vector search
SKILL_NS = "skillbox"                    # dedicated namespace; keeps skills out of other knowledge bases

SKILL_META = {}     # name -> {category, description, source_url, body, kb_id}
KB_TO_NAME = {}     # knowledge_base_id -> skill name (used to label query hits)

print("Skillbox backed by memorizz KnowledgeBase; namespace =", SKILL_NS)

Skillbox backed by memorizz KnowledgeBase; namespace = skillbox


### 8.2 · Seed a few skills

In production you'd sync these from a skills repo (the workshop pulls ~155 from `oracle/skills`).
Here we ingest a handful of synthetic AI‑engineering playbooks. `ingest_knowledge` embeds each one
through the provider and returns a `knowledge_base_id`; the re‑seed is idempotent because we clear
the namespace first.

In [33]:
SKILLS = [
    {"name": "agent/schema-discovery", "category": "agent",
     "description": "How to explore an unfamiliar Oracle schema before writing any SQL.",
     "source_url": "https://github.com/oracle/skills",
     "body": "# Schema discovery\n\n1. List tables: SELECT table_name FROM user_tables ORDER BY table_name.\n2. Inspect columns via user_tab_columns (name + data_type).\n3. Find relationships in user_constraints (type 'R' = foreign keys).\n4. Sample 5 rows per table before trusting column names.\n5. Only then write the query."},
    {"name": "rag/chunking-strategy", "category": "rag",
     "description": "Choosing a chunking strategy when ingesting a new corpus for retrieval.",
     "source_url": "https://github.com/oracle/skills",
     "body": "# Chunking strategy\n\n- Start with paragraph chunking + a 1000-char cap.\n- Use semantic chunking when topics drift mid-section.\n- Keep tables and code blocks as whole units.\n- Re-embed the whole corpus if you change model or dimension.\n- Validate with a 20-question retrieval set."},
    {"name": "vector/index-rebuild", "category": "vector",
     "description": "Rebuilding an Oracle AI Vector Search index safely in production.",
     "source_url": "https://github.com/oracle/skills",
     "body": "# Safe vector index rebuild\n\n1. Enter maintenance mode.\n2. Build the new index alongside the old one (never in place).\n3. Wait until new and old row counts match.\n4. Flip the read alias to the new index.\n5. Watch p99 for 15 min, then drop the old index."},
    {"name": "ops/incident-triage", "category": "ops",
     "description": "Triaging a production retrieval outage (SEV1) calmly and in order.",
     "source_url": "https://github.com/oracle/skills",
     "body": "# Incident triage\n\n1. Acknowledge the page within 5 minutes.\n2. Check service health and recent deploys first.\n3. If error rate > 5%, roll back before debugging.\n4. Post status to #incidents every 30 minutes.\n5. File a blameless postmortem within 48h."},
    {"name": "eval/prompt-regression-gate", "category": "eval",
     "description": "Setting up an evaluation gate before shipping a prompt or model change.",
     "source_url": "https://github.com/oracle/skills",
     "body": "# Prompt regression gate\n\n1. Freeze a labeled eval set (>= 50 cases) mirroring production.\n2. Score the current prompt for a baseline.\n3. Run the candidate; block on any metric regression beyond noise.\n4. Keep the eval set in version control next to the prompt.\n5. Promote every incident into a new eval case."},
]

# Idempotent re-seed: drop any skills previously ingested into this namespace.
try:
    prior = skill_kb.retrieve_knowledge_by_query("skill guide", namespace=SKILL_NS, limit=100)
    for kb_id in {h.get("knowledge_base_id") for h in prior if h.get("knowledge_base_id")}:
        skill_kb.delete_knowledge(kb_id)
except Exception:
    pass

SKILL_META.clear(); KB_TO_NAME.clear()
for s in SKILLS:
    # chunking_strategy="none" keeps each skill as a single retrievable document.
    kb_id = skill_kb.ingest_knowledge(corpus=s["body"], namespace=SKILL_NS, chunking_strategy="none")
    SKILL_META[s["name"]] = dict(s, kb_id=kb_id)
    KB_TO_NAME[kb_id] = s["name"]
print(f"Ingested {len(SKILL_META)} skills via memorizz KnowledgeBase (namespace '{SKILL_NS}').")

Ingested 5 skills via memorizz KnowledgeBase (namespace 'skillbox').


### 8.3 · The skill tools + manifest

Same three‑function API — `list_skills` (semantic search), `load_skill` (full body), and
`build_skill_manifest` (the always‑on top‑k block) — but every lookup now goes through
`KnowledgeBase`: no raw SQL, no separate embedding client.

In [34]:
def load_skill(name: str) -> str:
    """Load the full content of a named skill from the skillbox.
    Use this when the context's manifest lists a skill relevant to the current task.
    `name` is the full namespace, e.g. "agent/schema-discovery".
    """
    meta = SKILL_META.get(name)
    if not meta:
        return json.dumps({"error": f"no skill named {name!r}"})
    chunks = skill_kb.retrieve_knowledge(meta["kb_id"])          # round-trip through the provider
    body = "\n".join(c.get("content", "") for c in chunks) or meta.get("body", "")
    return json.dumps({"name": name, "category": meta.get("category"),
                       "description": meta.get("description"),
                       "source_url": meta.get("source_url"), "body": body})


def _skill_hits(query, k):
    """Distinct skill names for a query, scoped to the skillbox namespace via KnowledgeBase."""
    hits = skill_kb.retrieve_knowledge_by_query(query, namespace=SKILL_NS, limit=k)
    names, seen = [], set()
    for h in hits:
        name = KB_TO_NAME.get(h.get("knowledge_base_id"))
        if name and name not in seen:
            seen.add(name)
            names.append(name)
    return names


def list_skills(query: str, k: int = 5) -> str:
    """Search the skillbox semantically. Returns the top-k skills (name + description)."""
    rows = [{"name": n, "category": SKILL_META[n].get("category"),
             "description": SKILL_META[n].get("description")} for n in _skill_hits(query, k)]
    return json.dumps(rows)


def build_skill_manifest(query, k=3):
    """Top-k skills as a one-line-per-skill block, prepended to each turn's context."""
    try:
        names = _skill_hits(query, k)
    except Exception:
        return ""
    if not names:
        return ""
    lines = [f"  - {n} — {(SKILL_META[n].get('description') or '')[:240]}" for n in names]
    return ("Available skills (call load_skill(name) to read the full guide and follow it):\n"
            + "\n".join(lines) + "\n\n")

print("Skillbox ready (KnowledgeBase-backed): list_skills, load_skill, build_skill_manifest")

Skillbox ready (KnowledgeBase-backed): list_skills, load_skill, build_skill_manifest


In [35]:
# Semantic search over skills:
print("list_skills →", list_skills("I need to investigate a database I've never seen before", k=3))

# The always-on manifest that gets prepended to a turn:
print("\nmanifest →\n" + build_skill_manifest("how do I rebuild the vector index safely?", k=3))

# Load one skill's full body on demand:
skill = json.loads(load_skill("vector/index-rebuild"))
print("load_skill('vector/index-rebuild') →\n", skill["body"][:200], "...")

list_skills → [{"name": "agent/schema-discovery", "category": "agent", "description": "How to explore an unfamiliar Oracle schema before writing any SQL."}, {"name": "eval/prompt-regression-gate", "category": "eval", "description": "Setting up an evaluation gate before shipping a prompt or model change."}]

manifest →
Available skills (call load_skill(name) to read the full guide and follow it):
  - vector/index-rebuild — Rebuilding an Oracle AI Vector Search index safely in production.


load_skill('vector/index-rebuild') →
 # Safe vector index rebuild

1. Enter maintenance mode.
2. Build the new index alongside the old one (never in place).
3. Wait until new and old row counts match.
4. Flip the read alias to the new ind ...


### 8.4 · Wire the skillbox into a MemAgent

We register `load_skill` / `list_skills` as agent tools and prepend the manifest to each turn. The
agent now sees which skills exist and pulls the relevant one before answering.

In [36]:
skilled_agent = (
    MemAgentBuilder()
    .with_name("Memo-Skilled")
    .with_instruction(
        "You are Memo. The context may list available skills. When one is relevant, call "
        "load_skill(name), read its guide, and follow those steps in your answer."
    )
    .with_tool(load_skill)
    .with_tool(list_skills)
    .with_memory_provider(memory_core)
    .with_llm_config(LLM)
    .build()
)

def run_with_skills(agent, query, k=3):
    """Prepend the always-on skill manifest to the user's turn, then run the agent."""
    return agent.run(build_skill_manifest(query, k=k) + "User: " + query)

print(run_with_skills(
    skilled_agent,
    "I'm about to rebuild our Oracle vector index in production. Walk me through doing it safely."
))

Here’s a safe production runbook for rebuilding an Oracle AI Vector Search index.

## Safe Oracle vector index rebuild plan

### 1. Announce and enter maintenance mode

Before touching the index:

- Notify stakeholders.
- Pause or throttle writes if possible.
- Put the affected search path into maintenance/degraded mode.
- Confirm you have a rollback path.
- Capture current baseline metrics:
  - p50/p95/p99 latency
  - query error rate
  - index row count
  - table row count
  - CPU, memory, I/O
  - active sessions
  - application timeout rates

Do **not** rebuild the live index in place if production traffic depends on it.

---

### 2. Keep the old index serving traffic

The existing vector index should continue serving reads while the new one is built.

The safe pattern is:

```text
current live index  -> serves production reads
new index           -> built alongside it
```

Avoid:

```text
DROP old index
CREATE new index
```

That creates an outage or degraded search behavior if the

### 8.5 · The Oracle‑native variant — in‑database ONNX embeddings

The runnable cells above use memorizz's `KnowledgeBase` (embeddings go through the provider). The
original workshop instead works at the **raw‑SQL** level and embeds **inside the database** with a
loaded ONNX model (`ALL_MINILM_L12_V2`, 384‑dim), so retrieval needs **no external API call** and
data never leaves Oracle. A DBA loads the model once:

```sql
-- one-time, e.g. in bootstrap.py, as a privileged user
BEGIN
  DBMS_VECTOR.LOAD_ONNX_MODEL(
    directory  => 'DM_DUMP',
    file_name  => 'all_MiniLM_L12_v2.onnx',
    model_name => 'ALL_MINILM_L12_V2');
END;
/
```

Then the skill tools are exactly as in the
[workshop notebook](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/cb9ec507254ccf1e93b1540ce34e8bfbcf2c26c5/workshops/enterprise-data-agent-harness-workshop/workshop/notebook_complete.ipynb#L23)
— note `VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA)`, which computes the query embedding in SQL:

```python
@register
def tool_load_skill(name: str) -> str:
    """Load the full content of a named skill from the skillbox.
    Use this when the system prompt's manifest lists a skill relevant to the current task.
    `name` is the full namespace, e.g. "agent/schema-discovery".
    """
    with agent_conn.cursor() as cur:
        cur.execute("SELECT description, body, source_url, category FROM skillbox WHERE name = :n",
                    n=name)
        row = cur.fetchone()
    if not row:
        return json.dumps({"error": f"no skill named {name!r}"})
    desc, body, url, category = row
    body_text = body.read() if hasattr(body, "read") else str(body or "")
    return json.dumps({"name": name, "category": category, "description": desc,
                       "source_url": url, "body": body_text})


@register
def tool_list_skills(query: str, k: int = 5) -> str:
    """Search the skillbox semantically. Returns top-k skills (name + description)."""
    with agent_conn.cursor() as cur:
        cur.execute(
            "SELECT name, category, description FROM skillbox "
            f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
            " FETCH FIRST :k ROWS ONLY", q=query, k=k)
        hits = [{"name": n, "category": c, "description": d} for n, c, d in cur]
    return json.dumps(hits)


ALWAYS_ON_TOOLS.add("load_skill")


def build_skill_manifest(query, k=3):
    try:
        with agent_conn.cursor() as cur:
            cur.execute(
                "SELECT name, description FROM skillbox "
                f" ORDER BY VECTOR_DISTANCE(embedding, VECTOR_EMBEDDING({ONNX_EMBED_MODEL} USING :q AS DATA), COSINE) "
                " FETCH FIRST :k ROWS ONLY", q=query, k=k)
            rows = list(cur)
    except oracledb.DatabaseError:
        return ""
    if not rows:
        return ""
    lines = [f"  - {n} — {(d.read() if hasattr(d, 'read') else d)[:240]}" for n, d in rows]
    return ("Available skills (call load_skill(name) to read the full guide and follow it):\n"
            + "\n".join(lines) + "\n\n")


print(f"registered: load_skill, list_skills (TOOLS total: {len(TOOLS)})")
```

There, `@register` adds the function to the harness's `TOOLS` registry and `ALWAYS_ON_TOOLS` keeps
`load_skill` exposed on every turn — the equivalent of our `with_tool(...)` calls above. Same memory
pattern, two implementations: memorizz's `KnowledgeBase` (portable across the MongoDB / filesystem /
Oracle providers), or raw Oracle + in‑database ONNX (data never leaves the DB, zero external API calls).

**What just happened:** We added a memory type the framework doesn't ship — a **skillbox** — by
*composing* memorizz's `KnowledgeBase` primitive (no raw SQL needed). The manifest gives the agent
*awareness* of many skills for a few tokens; `load_skill` pulls the *full* guide only when needed.
That progressive‑disclosure pattern is how agents scale to large skill libraries without drowning the
context window.

> ### 📌 Key Takeaways — Part 8
> - A skillbox gives the agent **awareness of many skills for a few tokens**, loading the full guide only when relevant.
> - It's **not** a built‑in `MemoryType` — it's composed from the `KnowledgeBase` primitive, proving memory types are extensible.
> - Progressive disclosure is how agents scale to large skill libraries without flooding the context window.

---
# Part 9 · Short‑term memory I — semantic cache ⚡

> 💡 **Key Term — Semantic cache (`SEMANTIC_CACHE`):** Stores prior (query → answer) pairs as embeddings; a *semantically* similar new query returns the cached answer instantly, skipping the LLM.

> ➕ **Adding:** `SEMANTIC_CACHE` &nbsp;·&nbsp; *skip the LLM when a *semantically* similar question
> was already answered — big latency and cost wins.*

Not every memory is about recall; some is about **efficiency**. A **semantic cache** stores
prior (query → answer) pairs as embeddings. When a new query is *semantically* close enough to a
cached one (cosine similarity ≥ threshold), the cached answer is returned instantly — no LLM call.
This is short‑term/working memory in the engineering sense: a fast scratchpad in front of the model.

In [37]:
import time

cached_agent = (
    MemAgentBuilder()
    .with_instruction("You are a concise factual assistant.")
    .with_memory_provider(memory_core)
    .with_llm_config(LLM)
    .build()
)
cached_agent.enable_semantic_cache(threshold=0.6)

def timed(agent, q):
    t = time.time()
    a = agent.run(q)
    return time.time() - t, a

dt1, a1 = timed(cached_agent, "What is the capital of France?")
print(f"[{dt1:5.2f}s] cold   → {a1}")

# Different wording, same meaning → should hit the cache and skip the LLM.
dt2, a2 = timed(cached_agent, "Tell me France's capital city.")
print(f"[{dt2:5.2f}s] cached → {a2}")

[ 2.41s] cold   → Paris.
[ 0.51s] cached → Paris.


**What just happened:** The second query is phrased differently but means the same thing, so
it matches the cached embedding and returns far faster. Tune `threshold` (higher = stricter
matches, fewer false hits) and choose a `scope` (`local` to one agent, or shared). Caching is
ideal for FAQ‑style traffic; keep it off for queries that must always be fresh.

> ### 📌 Key Takeaways — Part 9
> - A semantic cache cuts **latency and cost** by reusing answers to *meaning‑equivalent* questions.
> - Tune `threshold` (higher = stricter matches) and `scope`; it's ideal for FAQ‑style traffic.
> - Keep it off for queries that must always be fresh.

---
# Part 10 · Working memory & episodic compression — summaries 🪟

> 💡 **Key Term — Working memory & summaries (`SHORT_TERM_MEMORY` + `SUMMARIES`):** The context window is the agent's finite workspace; summaries compress old episodic memory into compact rollups so long histories stay affordable.

> ➕ **Adding:** `SHORT_TERM_MEMORY` (context telemetry) + `SUMMARIES` &nbsp;·&nbsp; *keep long‑running
> agents inside the context budget.*

The context window is the agent's **working memory** — finite and the dominant cost driver. Two
tools keep it healthy: **telemetry** to see how full it is, and **summarization** to compress old
episodic memory into compact rollups (`SUMMARIES`) so history stays affordable without being lost.

In [38]:
stats = copilot.get_context_window_stats()
print("Context-window telemetry:", stats)

Context-window telemetry: {'timestamp': '2026-06-16T12:52:13.724077', 'prompt_tokens': 6845, 'completion_tokens': 90, 'total_tokens': 6935, 'context_window_tokens': 128000, 'percentage_used': 5.41796875, 'stage': 'iteration_2'}


In [39]:
# Compress older conversation turns into summaries (returns the new summary IDs).
summary_ids = copilot.generate_summaries(days_back=7, max_memories_per_summary=50)
print("Generated summary IDs:", summary_ids)

# The agent keeps a registry of summaries it can expand on demand.
for s in copilot.list_context_summaries():
    print("•", s)

Generated summary IDs: ['74a6bf1b-c4df-4415-b523-5d6f88988cb8']


**What just happened:** Summaries let an agent run for *months* of conversation without the
prompt growing without bound: old turns collapse into short, retrievable rollups, and the agent
can expand a summary back to detail when a topic resurfaces. `get_context_window_stats()` is your
window into how close any turn is to the limit — wire it into dashboards for production agents.

> ### 📌 Key Takeaways — Part 10
> - `get_context_window_stats()` is your **telemetry** on how full the context window is each turn.
> - `generate_summaries()` compresses old turns into retrievable rollups so long‑running agents stay in budget.
> - Working memory + summaries let an agent run for months without an unbounded prompt.

---
# Part 11 · Shared memory — multi‑agent coordination 🤝

> 💡 **Key Term — Shared memory (`SHARED_MEMORY`):** A blackboard multiple agents read and write — commands, reports, artifacts — so a team can coordinate over a common, traceable trail.

> ➕ **Adding:** `SHARED_MEMORY` &nbsp;·&nbsp; *a blackboard where multiple agents collaborate, delegate,
> and read each other's work.*

So far, one agent. Hard problems often want a **team**: a lead that decomposes the task and
specialists that do focused work. **Shared memory** is the blackboard they coordinate through — a
common store of commands, reports, and artifacts, with full traceability of who contributed what.

We'll stand up a lead orchestrator with two delegates (a researcher and a reviewer) and let the
`MultiAgentOrchestrator` run the collaboration over shared memory.

In [40]:
from memorizz.memagent.orchestrators import MultiAgentOrchestrator

def make_specialist(name, instruction):
    a = (MemAgentBuilder()
         .with_name(name)
         .with_instruction(instruction)
         .with_memory_provider(memory_core)
         .with_llm_config(LLM)
         .build())
    a.save()
    return a

researcher = make_specialist(
    "Researcher",
    "You research technical options and lay out concrete trade-offs (cost, quality, risk).")
reviewer = make_specialist(
    "Reviewer",
    "You critique proposals for correctness, security, and operational risk, then give a verdict.")

orchestrator = (
    MemAgentBuilder()
    .with_name("Lead")
    .with_instruction(
        "You are the lead engineer. Decompose the question, delegate to the Researcher and "
        "Reviewer, then synthesize one clear recommendation.")
    .with_memory_provider(memory_core)
    .with_llm_config(LLM)
    .with_delegates([researcher, reviewer])
    .build()
)
orchestrator.save()
print("Team ready: Lead + [Researcher, Reviewer]")

Team ready: Lead + [Researcher, Reviewer]


In [41]:
team = MultiAgentOrchestrator(root_agent=orchestrator, delegates=[researcher, reviewer])

try:
    result = team.execute_multi_agent_workflow(
        "Should we move our embeddings from text-embedding-3-small (256d) to a larger model? "
        "Weigh retrieval quality vs. cost and storage, and give a recommendation."
    )
    print(result)
except Exception as exc:
    print("Multi-agent run skipped/failed (it makes several LLM calls):", exc)

Certainly! Here’s a consolidated, decision-ready response to your question about whether to move from `text-embedding-3-small` (256d) to a larger embedding model, weighing retrieval quality versus cost and storage.

---

## Summary Recommendation

**Move to a larger embedding size—but do so conservatively, in incremental steps. Start by increasing dimensionality within your current model (e.g., `text-embedding-3-small` at 512d or 1024d), benchmark the impact, and only consider a model upgrade (e.g., `text-embedding-3-large`) if meaningful quality improvements justify the additional costs and complexity.**

---

## Comprehensive Analysis

### Technical and Cost Trade-offs

**1. Current Baseline:**  
- **`text-embedding-3-small` (256d)** is cheap, storage-efficient, and fast, but compressed to the point that nuanced or complex retrieval may suffer, especially in multi-lingual, technical, or domain-specific contexts.

**2. Increasing Dimensionality (same model):**  
- Upgrade from 256d to

In [42]:
# Peek at the blackboard the agents coordinated through.
try:
    print(team.get_shared_memory_context())
except Exception as exc:
    print("Shared-memory context is optional here:", exc)



---------SHARED MEMORY CONTEXT---------
Multi-agent coordination information:

Agent: 0f170b18-1932-436c-a30d-3e9f4edca9f1
Type: workflow_start
Content: {'original_query': 'Should we move our embeddings from text-embedding-3-small (256d) to a larger model? Weigh retrieval quality vs. cost and storage, and give a recommendation.', 'started_at': '2026-06-16T12:53:29.943369', 'orchestrator_type': 'root', 'delegate_count': 2}
Time: 2026-06-16T12:53:29.950360
---
Agent: 0f170b18-1932-436c-a30d-3e9f4edca9f1
Type: task_decomposition
Content: {'sub_tasks': [{'task_id': 'task_1', 'description': 'Research technical options for larger embedding models compared to text-embedding-3-small (256d), including available models, retrieval quality, cost, and storage requirements, and lay out concrete trade-offs between retrieval quality and operational costs.', 'assigned_agent_id': 'bcf83bb3-6fde-4472-9b02-9660ff2e79ad', 'priority': 1, 'dependencies': [], 'status': 'pending', 'result': None}, {'task_id'

**What just happened:** Multiple agents collaborated through one shared store — the lead
delegating, specialists reporting, everyone reading the same trail. `SHARED_MEMORY` is the
foundation for researcher/analyst/writer pipelines, human‑in‑the‑loop review queues, and the
built‑in **Deep Research** workflow (`from memorizz.memagent.orchestrators import DeepResearchWorkflow`).

> ### 📌 Key Takeaways — Part 11
> - Shared memory lets a **lead + specialists** collaborate through one common store via `MultiAgentOrchestrator`.
> - It's the foundation for researcher/analyst/writer pipelines and human‑in‑the‑loop review queues.
> - Every contribution is attributed and traceable on the blackboard.

---
# Part 12 · Putting it all together — application modes & the full copilot 🚀

> 💡 **Key Term — Application mode:** A preset (`assistant` / `workflow` / `deep_research`) that auto‑wires a sensible default *stack* of memory types so you don't assemble them by hand.

You've now added every memory type individually. In practice you rarely wire them by hand —
**application modes** pick a sensible default stack for you:

| Mode | Memory stack it enables |
|---|---|
| `assistant` | conversation · knowledge base · personas · entity · short‑term · summaries |
| `workflow` | workflow · toolbox · knowledge base · short‑term · summaries |
| `deep_research` | toolbox · shared · knowledge base · short‑term · summaries |

Here is **Memo‑Pro**: one agent combining persona + entity memory + knowledge base + semantic
cache under `assistant` mode — the full‑memory colleague we set out to build.

In [43]:
from memorizz.enums import ApplicationMode

memo_pro = (
    MemAgentBuilder()
    .with_name("Memo-Pro")
    .with_instruction("You are Memo-Pro, a fully memory-augmented engineering copilot for the team.")
    .with_application_mode(ApplicationMode.ASSISTANT.value)
    .with_persona(persona)
    .with_entity_memory(True)
    .with_memory_ids(copilot.memory_ids)  # share copilot scope: sees Ada, services, PagerPilot
    .with_semantic_cache(enabled=True, threshold=0.9)
    .with_memory_provider(memory_core)
    .with_llm_config(LLM)
    .build()
)
kb.attach_to_agent(memo_pro, kb_id)     # give it the runbook knowledge too
memo_pro.save()

print(memo_pro.run(
    "Using everything you know about Ada, our services, and the platform runbook: "
    "what are the top two things Ada should focus on this week, and why?"
))

1. **De-risk the `vector-search-svc` Oracle 23ai migration path end-to-end.**  
   **Why:** This service is on the critical RAG path, has a **99.9% SLA**, uses Oracle 23ai as the datastore, and depends on `text-embedding-3-small` with **256-dimensional embeddings**. Ada should validate schema, vector dimensions, index behavior, recall/latency, and rollback before relying on it in production.

2. **Rehearse and document the vector index rebuild procedure from the runbook.**  
   **Why:** The runbook rebuild is an operationally sensitive flow: put the service in maintenance mode, run  
   ```bash
   acme reindex --service vector-search-svc --batch 500
   ```  
   wait for old/new row counts to match, flip the alias, then leave maintenance mode. Since the expected rebuild time is about **40 minutes**, Ada should make sure PagerPilot’s dependency on retrieval can tolerate that window or has a safe maintenance/rollback plan.


**What just happened:** A single `run()` drew on **persona** (voice), **entity memory**
(Ada & the services), the **knowledge base** (runbook), **conversation memory** (the relationship
so far), and the **semantic cache** (speed) — all from one Oracle Memory Core. That's the payoff
of composable memory: each type is simple alone, but together they make an agent that feels like it
*knows* your project.

### Recap — the full memory map

| # | Memory type | Cognitive analogue | What it gave Memo | Key API |
|---|---|---|---|---|
| §2 | `CONVERSATION_MEMORY` | Episodic | Remembers the dialogue, even after restart | `agent.run()`, `MemAgent.load()` |
| §2 | `MEMAGENT` | — (durability) | The agent itself is persistable | `agent.save()` |
| §3 | `PERSONAS` | Semantic (identity) | A stable, evolvable voice | `Persona`, `set_persona()` |
| §4 | `ENTITY_MEMORY` | Semantic (facts) | Structured, queryable profile facts | `EntityMemory`, `with_entity_memory()` |
| §5 | `KNOWLEDGE_BASE` | Semantic (knowledge) | RAG over your documents | `KnowledgeBase.ingest_*()` |
| §6 | `TOOLBOX` | Procedural (skills) | Discoverable, durable tools | `Toolbox`, `with_tool()` |
| §6 | `TOOL_LOG` | Procedural (audit) | A replayable record of actions | `memory_manager.list_tool_logs()` |
| §7 | `WORKFLOW_MEMORY` | Procedural (processes) | Repeatable runbooks | `Workflow` |
| §8 | custom `skillbox` | Procedural (guides) | Skills to read & follow, loaded on demand | `build_skill_manifest()` / `load_skill()` |
| §9 | `SEMANTIC_CACHE` | Working (speed‑up) | Skips redundant LLM calls | `enable_semantic_cache()` |
| §10 | `SHORT_TERM_MEMORY` | Working memory | Context‑budget telemetry | `get_context_window_stats()` |
| §10 | `SUMMARIES` | Episodic (compressed) | Affordable long histories | `generate_summaries()` |
| §11 | `SHARED_MEMORY` | Social | Multi‑agent collaboration | `SharedMemory`, `MultiAgentOrchestrator` |

### Where to go next
- **Swap the provider, keep the code.** Everything here runs unchanged on `MongoDBProvider` or
  `FileSystemProvider` — memory types are decoupled from storage.
- **Multi‑tenant.** Pass `user_id=` to `run()` to isolate memory per end‑user with one agent.
- **Deep research.** `DeepResearchWorkflow` composes shared memory + tools + internet access.
- **Docs.** See `docs/memory-types/` and `docs/getting-started/concepts.md` in this repo.

### Cleanup (optional)
Run the cell below to close the Oracle connection pool when you're done. *(Re‑running earlier
cells afterward will require rebuilding `memory_core` in §0.6.)*

In [45]:
memory_core.close()
print("Memory Core connection pool closed.")

Memory Core connection pool closed.
